[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/05_gap_characterization.ipynb)

# R2-Q1 Week 4 — Characterize the PV → PD transfer gap

This is the last classroom notebook for R2-Q1. It picks up where
Notebook 04 left off.

Notebook 04 produced a headline number: your PV-trained classifier
loses some amount of accuracy when evaluated on PlantDoc rather than
PlantVillage, and the bootstrap confidence interval told you whether
that gap was real. What N04 did not tell you is *where* the gap lives
— which host species the model transfers cleanly to, which disease
categories it handles poorly, and which combinations drive most of
the failure.

This notebook decomposes the gap into two cuts:

1. **By host group.** Per-host accuracy across all disease categories.
   Answers: which plants does the model recognize cleanly when it
   sees them in a field photograph, and which does it stumble on?
2. **By disease category.** Per-category accuracy across all hosts.
   Answers: which disease categories transfer from lab to field, and
   which ones the model evidently learned in a PV-specific way?

By the end of this notebook you will have:

- A flat decomposition table (`gap_decomposition.parquet`) with one
  row per group: host or category name, sample count, accuracy, 95%
  bootstrap CI, and a flag for groups with too few samples to support
  a reliable confidence interval.
- A reader-facing summary (`characterization_summary.json`) naming
  the hosts and categories driving the gap and listing which cuts
  had enough data to be reliable.
- Two bar charts (`gap_by_host.png`, `gap_by_category.png`) with
  bootstrap error bars — the figures your Week 5 paper will lean on.

### What you need from N04

This notebook reads `pd_predictions.parquet` from your R2-Q1 output
directory (along with `precommit.json` for the disease-category
mapping). Both should already be there from running N04. Section 2
checks for them and fails with a clear error if either is missing.

### A note on compute

Unlike N04, this notebook does not require a GPU. There is no model
load and no inference pass — everything below is dataframe arithmetic
and bootstrap resampling on a few hundred rows. A free Colab CPU
runtime is fine.

### Install the iRI Lab package

If you opened this notebook in a fresh Colab session, you need to
install the package first. If you just finished running Notebook 04
in the same session, the package is already installed and you only
need to refresh it.

The diagnostic cell below tells you which of the two install patterns
in the following cell to uncomment. Run the diagnostic, read what it
prints, then uncomment exactly one line in the cell after it.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026 (no GPU required for this notebook),
# seed everything, and point OUTPUT_DIR at the R2-Q1 output directory.
import irilab2026 as iri

iri.setup(gpu_required=False)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

### Development mode

Unlike N04, this notebook doesn't have a multi-minute end-to-end
runtime. There's no model inference — just dataframe arithmetic and
bootstrap resampling on a few hundred rows. A full run takes well
under a minute even at the precommit's full bootstrap iteration count.

The `DEV_MODE` switch below still earns its keep on small iterations.
If you're tweaking a plot, fixing a typo, or debugging a save path,
clipping the bootstrap from a thousand iterations down to a hundred
shaves off most of the wait. The pattern also matches the one you
learned in Notebooks 03 and 04 — same toggle, same shape, one knob
fewer.

- **`DEV_MODE = True`**: cap the bootstrap at 100 iterations regardless
  of your precommit. The point estimates are still real, but the
  confidence intervals are wider than they would be at the precommit's
  full count. Numbers produced in dev mode are not paper-ready.
- **`DEV_MODE = False`**: honor your precommit's `n_bootstrap`. These
  are the numbers you report.

Ship the notebook with `DEV_MODE = False` so a student opening it
fresh produces real numbers. Flip to `True` only while you're
iterating on something.

In [ ]:
### 1.1) DEV_MODE switch

DEV_MODE = False

if DEV_MODE:
    BOOTSTRAP_CAP = 100      # max bootstrap iterations regardless of precommit
    print("⚠️  DEV_MODE is ON — numbers produced are NOT paper-ready.")
    print(f"    Bootstrap cap: {BOOTSTRAP_CAP} iterations")
else:
    BOOTSTRAP_CAP = None     # honor precommit's n_bootstrap
    print("DEV_MODE is OFF — full production run.")
    print(f"    Bootstrap cap: honor precommit n_bootstrap")